# 04 · 결과 확인
03이 반환한 보고서를 hash와 완료 상태 검증 후 읽습니다. 최신 결과를 자동 선택하지 않습니다.
[실행 안내](README.md) · [지표·보존 안내](LFW_PAIR_VERIFICATION.md)

In [ ]:
REPORT_DIR = None  # 03이 출력한 report_dir를 문자열로 지정
DISPLAY_MODEL = None  # None: 전체 모델, 또는 "arcface" 등
DISPLAY_FOLD = None  # None: 전체 folds
DISPLAY_TARGET_FMR = None  # None: 전체 목표; 예: 0.10
DISPLAY_ROWS = 24

In [ ]:
from pathlib import Path
import sys
import yaml
from IPython.display import display

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "research").is_dir() and (p / "configs").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Python:", sys.executable)

from research.experiments.lfw_pair_storage import read_report
if REPORT_DIR is None:
    print("03에서 생성한 REPORT_DIR를 첫 셀에 지정한 뒤 다시 실행하세요.")
else:
    report_path = (PROJECT_ROOT / REPORT_DIR).resolve()
    tables, manifest = read_report(report_path)
    print("상태:", manifest["status"], "· 완료:", manifest["completed_jobs"], "/", manifest["expected_jobs"])
    print("Report ID:", manifest["report_uid"])
    if manifest["status"] != "completed":
        print("부분 실행입니다. 전체 실험 완료 결과로 해석하지 마세요.")
    operating = tables["operating"]
    if DISPLAY_MODEL is not None:
        operating = operating.loc[operating.model.eq(DISPLAY_MODEL)]
    if DISPLAY_FOLD is not None:
        operating = operating.loc[operating.fold.eq(DISPLAY_FOLD)]
    if DISPLAY_TARGET_FMR is not None:
        operating = operating.loc[operating.target_fmr.eq(DISPLAY_TARGET_FMR)]
    print("필터 후 운영 결과 행:", len(operating), "· 표시 상한:", DISPLAY_ROWS)
    display(operating.head(DISPLAY_ROWS))
    paired = tables.get("paired", None)
    if paired is not None:
        for column, value in (("model", DISPLAY_MODEL), ("fold", DISPLAY_FOLD), ("target_fmr", DISPLAY_TARGET_FMR)):
            if value is not None and column in paired:
                paired = paired.loc[paired[column].eq(value)]
        print("동일 평가 조건의 보정 방법 비교 (전체 표는 analysis.zip):", len(paired))
        display(paired.head(DISPLAY_ROWS))
    display(tables["accuracy_cv"])
    print("전체 집계 표:", sorted(tables))
    print("해석 안내:", report_path / "INTERPRETATION.md")
    print("업로드 안내:", report_path / "START_HERE.md")
    print("집계 ZIP:", report_path / "analysis.zip")

target FMR와 realized FMR, 정수 성공/오류 수와 분모를 함께 확인합니다.
운영 TAR/FMR, nine-fold accuracy threshold의 Accuracy, test curve 진단은 서로 다른 결과입니다.
표시 필터와 head는 화면 표시만 제한합니다. 전체 모델·fold·seed 집계는 tables와 report에 유지됩니다.
같은 test를 공유하는 seed 반복을 독립 표본으로 취급하지 않습니다.